# Error direction — over- vs. under-estimation, per approach


**Metric — NMBE** (Normalized Mean Bias Error) = `mean(pred - actual) / mean(|actual|)`,
per (dataset, trim, series, approach[, regime]). Positive = systematic overestimation,
negative = underestimation.

**Secondary measure — `pct_over`**: the fraction of test days where prediction > actual.
Catches a model that averages out to ~0 bias while still oscillating wildly over/under.

**`unstable_scale`**: NMBE's denominator is `mean(|actual|)` — when a test period's
actual values are themselves near zero (e.g. a near-empty `concurrent_cases` window),
a routine-sized absolute error produces a wildly inflated NMBE that isn't a real "huge
bias," just a tiny denominator. Rows with `mean(|actual|) < 5` are flagged, not dropped —
`summarize_bias()`'s `median_nmbe` is the trustworthy central-tendency number whenever
`n_unstable > 0` for an approach.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from error_direction import (
    build_bias_table, summarize_bias, mae_comparison_table, APPROACH_ORDER, PPM_MODELS,
    PPM_REGIMES, DISPLAY_NAMES, REGIME_DISPLAY, synth_logs,
)
from split_correlation import real_logs

RESULTS = ROOT.parent / 'results'
print(f'Approaches: {len(APPROACH_ORDER)}   PPM regimes: {PPM_REGIMES}')
print(f'Synthetic logs: {len(synth_logs)}   Real-life logs: {len(real_logs)}')

Approaches: 22   PPM regimes: ['plain_field', 'full', 'half']
Synthetic logs: 10   Real-life logs: 8


In [2]:
import matplotlib.colors as mcolors

def load_or_build(cache_name, **build_kwargs):
    cache = RESULTS / cache_name
    if cache.exists():
        df = pd.read_csv(cache)
        print(f'[cache] loaded {cache}  ({len(df)} rows)')
    else:
        df = build_bias_table(**build_kwargs)
        df.to_csv(cache, index=False)
        print(f'[built] saved -> {cache}  ({len(df)} rows)')
    return df


def show_coverage(df, label):
    cov = (df.assign(label=df.apply(
            lambda r: f"{DISPLAY_NAMES.get(r['approach'], r['approach'])}"
                      + (f" ({REGIME_DISPLAY[r['regime']]})" if pd.notna(r['regime']) else ''),
            axis=1))
          .groupby(['label', 'series'])['nmbe'].apply(lambda s: s.notna().sum())
          .unstack('series'))
    order = [f"{DISPLAY_NAMES.get(a, a)}" + (f" ({REGIME_DISPLAY[r]})" if a in PPM_MODELS else '')
             for a in APPROACH_ORDER for r in (PPM_REGIMES if a in PPM_MODELS else [None])]
    cov = cov.reindex([o for o in order if o in cov.index])
    print(f'(out of {df[["dataset","trim"]].drop_duplicates().shape[0]} dataset/trim combos, {label})')
    return cov


def show_mismatches(df):

    checked = df[df['recorded_mae'].notna()]
    mismatches = checked[checked['mae_match'] == False]
    print(f'{len(checked)}/{len(df)} rows have a recorded MAE to check against; '
          f'{len(mismatches)} MISMATCH')
    if len(mismatches):
        return (mismatches[['dataset', 'trim', 'approach', 'regime', 'series', 'mae', 'recorded_mae']]
               .sort_values(['approach', 'regime', 'dataset']).reset_index(drop=True))
    return mismatches


def show_unstable(df):
    unstable = df[df['unstable_scale']]
    print(f'{len(unstable)}/{len(df)} rows flagged unstable (mean(|actual|) < 5)')
    if len(unstable):
        return (unstable[['dataset', 'trim', 'series', 'actual_scale']]
               .drop_duplicates().sort_values('actual_scale').reset_index(drop=True))
    return unstable


_DIVERGING_CMAP = mcolors.LinearSegmentedColormap.from_list(
    'error_direction_diverging', ['#2a78d6', '#f0efec', '#e34948'])


def plot_bias(summary, title, save_name, tick_fontsize=16, label_fontsize=16,
             title_fontsize=22, value_fontsize=13):

    plot_df = summary.reset_index()
    vmax = max(plot_df['median_nmbe'].abs().max(), 1e-6)
    norm = (plot_df['median_nmbe'].clip(-vmax, vmax) / vmax + 1) / 2
    colors = _DIVERGING_CMAP(norm)

    fig, ax = plt.subplots(figsize=(max(12, len(plot_df) * 0.7), 7.5))
    x = np.arange(len(plot_df))
    bars = ax.bar(x, plot_df['median_nmbe'], color=colors)
    ax.axhline(0, color='#0b0b0b', linewidth=1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['approach'], rotation=60, ha='right',
                       fontsize=tick_fontsize, fontweight='bold')
    ax.tick_params(axis='y', labelsize=tick_fontsize)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight('bold')
    ax.set_ylabel('median NMBE\n(negative = underestimate, positive = overestimate)',
                 fontsize=label_fontsize, fontweight='bold')
    ax.set_title(title, fontsize=title_fontsize, fontweight='bold')

    ymin, ymax = plot_df['median_nmbe'].min(), plot_df['median_nmbe'].max()
    span = max(ymax - ymin, 1e-6)
    pad = span * 0.03
    for bar, val in zip(bars, plot_df['median_nmbe']):
        ypos = val + pad if val >= 0 else val - pad
        va = 'bottom' if val >= 0 else 'top'
        label = f'{val:+.2f}'
        if label in ('+0.00', '-0.00'):
            label = '0.00'
        ax.text(bar.get_x() + bar.get_width() / 2, ypos, label,
                ha='center', va=va, fontsize=value_fontsize, fontweight='bold',
                color='#0b0b0b', rotation=90)
    ax.set_ylim(ymin - span * 0.25, ymax + span * 0.25)

    ax.grid(axis='y', color='#e1e0d9', linewidth=1, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

    fig.canvas.draw()
    fig.savefig(RESULTS / save_name, dpi=150, bbox_inches='tight', pad_inches=0.2)
    plt.show()


_FAMILY_COLOR = {
    'baseline': '#8a8a86', 'statistical': '#2a78d6', 'ml': '#eb6834',
    'chronos': '#4a3aa7', 'tabpfn': '#008300', 'ppm': '#e34948',
}
_PPM_REGIME_ALPHA = {'plain_field': 0.45, 'half': 0.7, 'full': 1.0}
_BASELINE_LABELS = {'Naive', 'Val. Avg.'}
_STAT_LABELS = {'Prophet', 'ETS', 'SARIMAX', 'STL'}


def _mae_bar_color_alpha(approach: str, regime):
    if approach == 'Chronos':
        return _FAMILY_COLOR['chronos'], 1.0
    if approach == 'TabPFN':
        return _FAMILY_COLOR['tabpfn'], 1.0
    if pd.notna(regime):
        return _FAMILY_COLOR['ppm'], _PPM_REGIME_ALPHA.get(regime, 1.0)
    if approach in _BASELINE_LABELS:
        return _FAMILY_COLOR['baseline'], 1.0
    if approach in _STAT_LABELS:
        return _FAMILY_COLOR['statistical'], 1.0
    return _FAMILY_COLOR['ml'], 1.0


def plot_mae_comparison(df, series_key, series_label, title, save_name=None):
    plot_df = df[df[f'{series_key}_mae'].notna()].reset_index(drop=True)
    labels = [row.approach + (f' ({REGIME_DISPLAY[row.regime]})' if pd.notna(row.regime) else '')
             for row in plot_df.itertuples()]
    colors, alphas = zip(*(_mae_bar_color_alpha(row.approach, row.regime)
                          for row in plot_df.itertuples()))

    fig, ax = plt.subplots(figsize=(max(12, len(labels) * 0.6), 6))
    x = np.arange(len(labels))
    for xi, val, color, alpha in zip(x, plot_df[f'{series_key}_mae'], colors, alphas):
        ax.bar(xi, val, color=color, alpha=alpha, edgecolor='white', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=12)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight('bold')
    ax.set_ylabel(f'{series_label} MAE', fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.set_title(title, fontsize=17, fontweight='bold')
    ax.grid(axis='y', alpha=0.25)
    fig.canvas.draw()
    if save_name:
        fig.savefig(RESULTS / save_name, dpi=150, bbox_inches='tight', pad_inches=0.2)
    plt.show()
    return fig


# Part 1 — Synthetic logs

Full Chronos + TabPFN coverage; Camargo/PGTNet/PT_RT/PT_su across all three PPM regimes where raw predictions exist on disk.

In [ ]:
df_synth = load_or_build('error_direction_synthetic.csv', is_real=False)

## Coverage

In [ ]:
show_coverage(df_synth, 'synthetic')

## Sanity check — computed MAE vs recorded metrics

In [ ]:
show_mismatches(df_synth)

## Unstable-normalization rows

In [ ]:
show_unstable(df_synth)

## Summary — mean/median NMBE and mean %-days-over, per approach

In [ ]:
summary_synth = summarize_bias(df_synth)
summary_synth.round(3)

## Summary split by series

In [ ]:
for series_name in ['concurrent_cases', 'throughput_time']:
    print(f'=== {series_name} ===')
    print(summarize_bias(df_synth[df_synth['series'] == series_name]).round(3).to_string())
    print()


## Plot

In [ ]:
plot_bias(summary_synth, 'Synthetic logs — median error direction per approach', 'error_direction_synthetic.pdf')

## Plot — concurrent cases only, every approach

In [ ]:
summary_synth_cc_all = summarize_bias(df_synth[df_synth['series'] == 'concurrent_cases'])
plot_bias(summary_synth_cc_all,
         'Synthetic logs — median error direction, concurrent cases (all approaches)',
         'error_direction_synthetic_cc_all.pdf')
summary_synth_cc_all.round(3)

In [ ]:
from error_direction import get_pmsd_bias_rows
from ts_comparison import load_splits

pmsd_rows = []
for ds in synth_logs:
    try:
        split = load_splits(ds, 'none', False)
    except Exception as e:
        print(f'  split ERROR {ds}: {e}')
        continue
    pmsd_rows.extend(get_pmsd_bias_rows(ds, 'none', False, split))

df_pmsd_synth = pd.DataFrame(pmsd_rows)
n_ds = df_pmsd_synth['dataset'].nunique() if len(df_pmsd_synth) else 0
print(f'{len(df_pmsd_synth)} PMSD rows computed ({n_ds}/{len(synth_logs)} datasets)')


n_existing = (df_synth['approach'] == 'pmsd').sum()
if n_existing:
    print(f'[note] df_synth cache already had {n_existing} pmsd rows -- dropping before merge')
df_synth_pmsd = pd.concat(
    [df_synth[df_synth['approach'] != 'pmsd'], df_pmsd_synth], ignore_index=True)

In [ ]:

TABLE_IV_ORDER = [
    'Naive', 'Val. Avg.',
    'Prophet', 'ETS', 'SARIMAX', 'STL',
    'Ridge', 'Ridge_M', 'N-BEATS', 'TFT',
    'Chronos', 'TabPFN',
    'PMSD', 'Simod', 'AgentSimulator',
    'GLSTM (Plain)', 'PGT (Plain)', 'PT_RT (Plain)', 'PT_su (Plain)',
    'GLSTM (First)', 'PGT (First)', 'PT_su (First)', 'PT_RT (First)',
    'GLSTM (Half)', 'PGT (Half)', 'PT_su (Half)', 'PT_RT (Half)',
]


_LATEX_RELABEL = {
    'Naive': 'Naïve',
    'Ridge_M': r'Ridge$_{\mathrm{M}}$',
    'PT_RT': r'PT$_{\mathrm{RT}}$',
    'PT_su': r'PT$_{\mathrm{su}}$',
    'AgentSimulator': 'ASIM',
}


def _latex_label(label: str) -> str:
    for k, v in _LATEX_RELABEL.items():
        if label.startswith(k):
            return label.replace(k, v, 1)
    return label


summary_table_iv_cc = summarize_bias(df_synth_pmsd[df_synth_pmsd['series'] == 'concurrent_cases'])
missing = [a for a in TABLE_IV_ORDER if a not in summary_table_iv_cc.index]
if missing:
    print(f'[warn] no NMBE data for: {missing}')
summary_table_iv_cc = summary_table_iv_cc.reindex([a for a in TABLE_IV_ORDER if a in summary_table_iv_cc.index])

summary_table_iv_cc = summary_table_iv_cc.rename(index=_latex_label)
summary_table_iv_cc.round(3)

In [ ]:
import re

TABLE_ORDER = [
    'Naive', 'Val. Avg.',
    'Prophet', 'ETS', 'SARIMAX', 'STL',
    'Ridge', 'Ridge_M', 'N-BEATS', 'TFT',
    'Chronos', 'TabPFN',
    'PMSD', 'Simod', 'AgentSimulator',
    'GLSTM (Plain)', 'PGT (Plain)', 'PT_RT (Plain)', 'PT_su (Plain)',
    'GLSTM (First)', 'PGT (First)', 'PT_su (First)', 'PT_RT (First)',
    'GLSTM (Half)', 'PGT (Half)', 'PT_su (Half)', 'PT_RT (Half)',
]


TABLE_RELABEL = {
    'Naive': 'Naïve',
    'Ridge_M': r'Ridge$_{\mathrm{M}}$',
    'PT_RT': r'PT$_{\mathrm{RT}}$',
    'PT_su': r'PT$_{\mathrm{su}}$',
    'AgentSimulator': 'ASIM',
}


def _table_label(label: str, strip_regime: bool = True) -> str:
    if strip_regime:
        label = re.sub(r' \((Plain|First|Half)\)$', '', label)
    for k, v in TABLE_RELABEL.items():
        if label.startswith(k):
            return label.replace(k, v, 1)
    return label


TOP_GROUP_OF = {
    'Naive': 'Basel.', 'Val. Avg.': 'Basel.',
    'Prophet': 'Statistical', 'ETS': 'Statistical', 'SARIMAX': 'Statistical', 'STL': 'Statistical',
    'Ridge': 'ML', 'Ridge_M': 'ML', 'N-BEATS': 'ML', 'TFT': 'ML',
    'Chronos': 'FM', 'TabPFN': 'FM',
    'PMSD': 'SIM', 'Simod': 'SIM', 'AgentSimulator': 'SIM',
}


def _top_group_for(raw_label: str) -> str:
    base = raw_label.split(' (')[0]
    if base in ('GLSTM', 'PGT', 'PT_RT', 'PT_su'):
        return 'Trace-Level PPM'
    return TOP_GROUP_OF[base]


def _sub_group_for(raw_label: str):
    """Plain/First/Half sub-group -- only meaningful (non-None) inside Trace-Level PPM,
    since that's the only family with a '(regime)' suffix in its raw label."""
    if '(' in raw_label:
        return raw_label.split('(')[1].rstrip(')')
    return None


def _bounds(seq):
    """Yield (lo, hi, value) for each maximal run of equal values in seq."""
    lo = 0
    for i in range(1, len(seq) + 1):
        if i == len(seq) or seq[i] != seq[lo]:
            yield lo, i, seq[lo]
            lo = i


def plot_bias_grouped2(summary_raw, order, save_name, gap=0.9, subgap=0.4,
                       tick_fontsize=14, label_fontsize=15, group_fontsize=13,
                       subgroup_fontsize=11, value_fontsize=11):
    """Two-level grouped bar chart of median NMBE, bar color encodes bias direction."""
    order = [a for a in order if a in summary_raw.index]
    top_groups = [_top_group_for(a) for a in order]
    sub_groups = [_sub_group_for(a) for a in order]
    vals = summary_raw.loc[order, 'median_nmbe'].to_numpy()

    xs = [0.0]
    for i in range(1, len(order)):
        if top_groups[i] != top_groups[i - 1]:
            step = 1 + gap
        elif sub_groups[i] != sub_groups[i - 1]:
            step = 1 + subgap
        else:
            step = 1
        xs.append(xs[-1] + step)
    xs = np.array(xs)

    vmax = max(np.abs(vals).max(), 1e-6)
    norm = (np.clip(vals, -vmax, vmax) / vmax + 1) / 2
    colors = _DIVERGING_CMAP(norm)

    fig, ax = plt.subplots(figsize=(max(14, len(order) * 0.62), 7.5))
    ax.bar(xs, vals, color=colors, width=0.85)
    ax.axhline(0, color='#0b0b0b', linewidth=1.2)

    labels = [_table_label(a) for a in order]
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=tick_fontsize, fontweight='bold')
    ax.tick_params(axis='y', labelsize=tick_fontsize)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight('bold')
    ax.set_ylabel('median NMBE', fontsize=label_fontsize, fontweight='bold')

    ymin, ymax = vals.min(), vals.max()
    span = max(ymax - ymin, 1e-6)
    pad = span * 0.03
    for x, val in zip(xs, vals):
        ypos = val + pad if val >= 0 else val - pad
        va = 'bottom' if val >= 0 else 'top'
        label = f'{val:+.2f}'
        if label in ('+0.00', '-0.00'):
            label = '0.00'
        ax.text(x, ypos, label, ha='center', va=va, fontsize=value_fontsize,
                fontweight='bold', color='#0b0b0b', rotation=90)
    ax.set_ylim(ymin - span * 0.34, ymax + span * 0.30)

    trans = ax.get_xaxis_transform()
    top_bounds = list(_bounds(top_groups))
    for lo, hi, name in top_bounds:
        if lo > 0:
            sep_x = (xs[lo - 1] + xs[lo]) / 2
            ax.axvline(sep_x, color='#9a9990', linewidth=1.4, ymin=0, ymax=1, zorder=0)
        center = (xs[lo] + xs[hi - 1]) / 2
        ax.text(center, 1.10, name, transform=trans, ha='center', va='bottom',
                fontsize=group_fontsize, fontweight='bold', color='#3a3a36')

    combined = list(zip(top_groups, sub_groups))
    for lo, hi, (top, sub) in _bounds(combined):
        if sub is None:
            continue
        if lo > 0 and sub_groups[lo - 1] is not None:
            sep_x = (xs[lo - 1] + xs[lo]) / 2
            ax.axvline(sep_x, color='#d8d7d0', linewidth=1, ymin=0, ymax=1, zorder=0)
        center = (xs[lo] + xs[hi - 1]) / 2
        ax.text(center, 1.02, sub, transform=trans, ha='center', va='bottom',
                fontsize=subgroup_fontsize, fontweight='bold', color='#6a6a63', style='italic')

    ax.set_xlim(xs[0] - 0.7, xs[-1] + 0.7)
    ax.grid(axis='y', color='#e1e0d9', linewidth=1, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

    fig.canvas.draw()
    fig.savefig(RESULTS / save_name, dpi=150, bbox_inches='tight', pad_inches=0.3)
    plt.show()


summary_table_cc = summarize_bias(df_synth_pmsd[df_synth_pmsd['series'] == 'concurrent_cases'])
plot_bias_grouped2(summary_table_cc, TABLE_ORDER, 'error_direction_table_grouped_cc.pdf')

## Same, throughput time

In [ ]:
summary_table_tt = summarize_bias(df_synth_pmsd[df_synth_pmsd['series'] == 'throughput_time'])
plot_bias_grouped2(summary_table_tt, TABLE_ORDER, 'error_direction_table_grouped_tt.pdf')

## Full row-level table

In [ ]:
pd.set_option('display.max_rows', 200)
df_synth.sort_values(['approach', 'regime', 'dataset']).reset_index(drop=True)

# Part 2 SSD

In [19]:
df_real_ssd = load_or_build('error_direction_ssd.csv', trims=['ssd'], is_real=True)

[cache] loaded /Users/mi98gr/Documents/PhD/27_coding_local/06_System_level_prediction/results/error_direction_ssd.csv  (448 rows)


## Coverage

In [ ]:
show_coverage(df_real_ssd, 'ssd')

## Sanity check — computed MAE vs recorded metrics

In [ ]:
show_mismatches(df_real_ssd)

## Unstable-normalization rows

In [ ]:
show_unstable(df_real_ssd)

## Combined table — CC/TT x synthetic/real-life(ssd), every approach, as CSV


In [ ]:
def _bias_row(df, series_key):
    return summarize_bias(df[df['series'] == series_key])['median_nmbe'].reindex(TABLE_ORDER)

error_direction_table = pd.DataFrame({
    'Synthetic — CC':       _bias_row(df_synth_pmsd, 'concurrent_cases'),
    'Synthetic — TT':       _bias_row(df_synth_pmsd, 'throughput_time'),
    'Real-life (ssd) — CC': _bias_row(df_real_ssd, 'concurrent_cases'),
    'Real-life (ssd) — TT': _bias_row(df_real_ssd, 'throughput_time'),
}).T
error_direction_table.index.name = 'domain_series'
error_direction_table.columns.name = 'approach'

out_path = RESULTS / 'error_direction_table_all.csv'
error_direction_table.round(3).to_csv(out_path)
print(f'saved -> {out_path}  ({error_direction_table.shape[0]} rows x {error_direction_table.shape[1]} approaches, values = median NMBE)')
error_direction_table.round(3)